In [1]:
import numpy as np
import pandas as pd

In [2]:
l01_fold0 = pd.read_csv(f"/scratch1/smaruj/generate_genomic_dot/lambda/lambda_0.1/fold0_1.0_genomic_windows_table_results.tsv", sep="\t")
l01_fold0["fold"] = [0 for i in range(len(l01_fold0))]
l01_fold0["lambda"] = [0.1 for i in range(len(l01_fold0))]

In [3]:
l01_fold1 = pd.read_csv(f"/scratch1/smaruj/generate_genomic_dot/lambda/lambda_0.1/fold1_1.0_genomic_windows_table_results.tsv", sep="\t")
l01_fold1["fold"] = [1 for i in range(len(l01_fold1))]
l01_fold1["lambda"] = [0.1 for i in range(len(l01_fold1))]

In [4]:
l01_fold2 = pd.read_csv(f"/scratch1/smaruj/generate_genomic_dot/lambda/lambda_0.1/fold2_1.0_genomic_windows_table_results.tsv", sep="\t")
l01_fold2["fold"] = [2 for i in range(len(l01_fold2))]
l01_fold2["lambda"] = [0.1 for i in range(len(l01_fold2))]

In [5]:
l1_fold0 = pd.read_csv(f"/scratch1/smaruj/generate_genomic_dot/lambda/lambda_1.0/fold0_1.0_genomic_windows_table_results.tsv", sep="\t")
l1_fold0["fold"] = [0 for i in range(len(l1_fold0))]
l1_fold0["lambda"] = [1.0 for i in range(len(l1_fold0))]

In [6]:
l1_fold1 = pd.read_csv(f"/scratch1/smaruj/generate_genomic_dot/lambda/lambda_1.0/fold1_1.0_genomic_windows_table_results.tsv", sep="\t")
l1_fold1["fold"] = [1 for i in range(len(l1_fold1))]
l1_fold1["lambda"] = [1.0 for i in range(len(l1_fold1))]

In [7]:
l1_fold2 = pd.read_csv(f"/scratch1/smaruj/generate_genomic_dot/lambda/lambda_1.0/fold2_1.0_genomic_windows_table_results.tsv", sep="\t")
l1_fold2["fold"] = [2 for i in range(len(l1_fold2))]
l1_fold2["lambda"] = [1.0 for i in range(len(l1_fold2))]

In [8]:
l10_fold0 = pd.read_csv(f"/scratch1/smaruj/generate_genomic_dot/lambda/lambda_10.0/fold0_1.0_genomic_windows_table_results.tsv", sep="\t")
l10_fold0["fold"] = [0 for i in range(len(l10_fold0))]
l10_fold0["lambda"] = [10.0 for i in range(len(l10_fold0))]

In [9]:
l10_fold1 = pd.read_csv(f"/scratch1/smaruj/generate_genomic_dot/lambda/lambda_10.0/fold1_1.0_genomic_windows_table_results.tsv", sep="\t")
l10_fold1["fold"] = [1 for i in range(len(l10_fold1))]
l10_fold1["lambda"] = [10.0 for i in range(len(l10_fold1))]

In [10]:
l10_fold2 = pd.read_csv(f"/scratch1/smaruj/generate_genomic_dot/lambda/lambda_10.0/fold2_1.0_genomic_windows_table_results.tsv", sep="\t")
l10_fold2["fold"] = [2 for i in range(len(l10_fold2))]
l10_fold2["lambda"] = [10.0 for i in range(len(l10_fold2))]

In [11]:
df = pd.concat([l01_fold0, l01_fold1, l01_fold2,
                l1_fold0, l1_fold1, l1_fold2,
                l10_fold0, l10_fold1, l10_fold2], ignore_index=True)

In [12]:
# summing edits together
df["num_edits"] = df["num_edits_slice0"] + df["num_edits_slice1"]

In [13]:
# optimizations with no edits
counts = df.groupby("lambda")["num_edits"].apply(lambda x: (x == 0).sum())
print(counts)

lambda
0.1     0
1.0     0
10.0    0
Name: num_edits, dtype: int64


In [14]:
# eliminating them
df = df[df["num_edits"] > 0]

In [18]:
df["dot7_result"]

0      1.112761
1      0.579329
2      1.162761
3      1.073372
4      1.216058
         ...   
487    0.566590
488    1.103421
489    0.727131
490    1.228244
491    1.242028
Name: dot7_result, Length: 492, dtype: float64

In [19]:
# optimizations with edits but not sufficent dot score
dot_counts = df.groupby("lambda")["dot7_result"].apply(lambda x: (x <= 0.05).sum())
print(dot_counts)

lambda
0.1     0
1.0     0
10.0    1
Name: dot7_result, dtype: int64


In [21]:
len(df[df["dot7_result"] <= 0.05])

1

In [22]:
# eliminating them
df = df[df["dot7_result"] > 0.05]

In [23]:
# successful optimizations only, average number of edits
avg_num_edits = df.groupby("lambda")["num_edits"].mean()
print(avg_num_edits)

lambda
0.1     1158.829268
1.0     1049.884146
10.0     595.411043
Name: num_edits, dtype: float64


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df["dot_diff"] = df["dot_result"] - df["dot_init"]

In [ ]:
df["CTCFs_num"] = df["CTCFs_num_slice0"] + df["CTCFs_num_slice1"]

In [ ]:
df["FIMO_sum"] = df["FIMO_sum_slice0"] + df["FIMO_sum_slice1"]

In [ ]:
df["FIMO_max"] = df[["FIMO_max_slice0", "FIMO_max_slice1"]].max(axis=1)

In [ ]:
order = sorted(df["lambda"].unique())

plt.figure(figsize=(6,4))
sns.boxplot(data=df, x="lambda", y="dot_diff", order=order)
plt.xlabel("lambda")
plt.ylabel("dot_diff")
plt.title("dot_diff by lambda")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x="lambda", y="PearsonR", order=order)
plt.xlabel("lambda")
plt.ylabel("PearsonR")
plt.title("PearsonR by lambda")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x="lambda", y="CTCFs_num", order=order)
plt.xlabel("lambda")
plt.ylabel("CTCFs_num")
plt.title("CTCFs_num by lambda")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x="lambda", y="FIMO_sum", order=order)
plt.xlabel("lambda")
plt.ylabel("FIMO_sum")
plt.title("FIMO_sum by lambda")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x="lambda", y="FIMO_max", order=order)
plt.xlabel("lambda")
plt.ylabel("FIMO_max")
plt.title("FIMO_max by lambda")
plt.tight_layout()
plt.show()